In [20]:
# First we import some libraries. 
import pathlib
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import geopandas as gpd
import pandas as pd
import tqdm

In [21]:
# Local stable version of popexposure
from popexposure.pop_estimator import PopEstimator

In [22]:
# set directories 
base_path = pathlib.Path.cwd().parent.parent

pop_dat_dir = base_path / "GHSL" / "1km" 

ghsl_2000 = pop_dat_dir / "GHS_POP_E2000_GLOBE_R2023A_54009_1000_V1_0" / "GHS_POP_E2000_GLOBE_R2023A_54009_1000_V1_0.tif"
ghsl_2005 = pop_dat_dir / "GHS_POP_E2005_GLOBE_R2023A_54009_1000_V1_0" / "GHS_POP_E2005_GLOBE_R2023A_54009_1000_V1_0.tif"
ghsl_2010 = pop_dat_dir / "GHS_POP_E2010_GLOBE_R2023A_54009_1000_V1_0" / "GHS_POP_E2010_GLOBE_R2023A_54009_1000_V1_0.tif"
ghsl_2015 = pop_dat_dir / "GHS_POP_E2015_GLOBE_R2023A_54009_1000_V1_0" / "GHS_POP_E2015_GLOBE_R2023A_54009_1000_V1_0.tif"
ghsl_2020 = pop_dat_dir / "GHS_POP_E2020_GLOBE_R2023A_54009_1000_V1_0" / "GHS_POP_E2020_GLOBE_R2023A_54009_1000_V1_0.tif"


all_wf_dat = base_path / "national_wf_disaster_hosp" / "local_data" / "monthly_worst_100_fires_exposure"


zctas_2020 = base_path / "national_wf_disaster_hosp" / "local_data" / "raw_data" / "zctas_2020.parquet"

In [23]:
# make a list of paths that we're going to use for each month 
ghsl_paths = [ghsl_2000, ghsl_2005, ghsl_2010, ghsl_2015, ghsl_2020]

# rep pattern
rep_pattern = [3*12, 5*12 ,5*12 ,5*12 ,1*12]

# list of ghsls to use for months
repeated_paths = [path for path, count in zip(ghsl_paths, rep_pattern) for _ in range(count)]


In [24]:
all_wf_exposure = sorted([all_wf_dat / file for file in os.listdir(all_wf_dat) if 'month' in file])
print(all_wf_exposure)

[PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-01-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-02-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-03-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-04-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-05-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-06-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/monthly_worst_100_fires_exposure/month_2000-07-01.geojson'), PosixPath('/Volumes/squirrel-utopia/national_wf_disaster_hosp

In [25]:
est = PopEstimator(admin_data=zctas_2020)

In [26]:
exposed_pop_df = est.est_exposed_pop(hazard_data=all_wf_exposure[0],
                                     hazard_specific=False,
                                     pop_data=repeated_paths[0])

In [27]:
print(exposed_pop_df)

None


In [ ]:
est = PopEstimator(admin_data=zctas_2020)
exposed_pop = []

for i in tqdm.tqdm(range(len(all_wf_exposure)), desc="Calculating exposed population"):
    exposed_pop_df = est.est_exposed_pop(
        hazard_data=all_wf_exposure[i],
        hazard_specific=False,
        pop_data=repeated_paths[i])
    if exposed_pop_df is not None:
        exposed_pop_df["month"] = i + 1  # add month column
        exposed_pop.append(exposed_pop_df)

Calculating exposed population: 100%|██████████| 228/228 [04:40<00:00,  1.23s/it]  


AttributeError: 'list' object has no attribute 'to_csv'

In [31]:
final_df = pd.concat(exposed_pop, ignore_index=True)
output_path = "/Volumes/squirrel-utopia/national_wf_disaster_hosp/local_data/intermediate_data/exposed_population_counts_worst_100_fires.csv"
final_df.to_csv(output_path, index=False)
